### E-Commerce Recommendation System

#### Milestone 1

In [1]:
import pandas as pd

# Define chunk size to avoid memory issues
chunk_size = 10_000

interaction_chunks = []

# Map event types to numeric interaction scores
event_strength = {'view': 1, 'addtocart': 2, 'transaction': 3}

chunks = pd.read_csv('/content/events.csv', chunksize=chunk_size)

for i, chunk in enumerate(chunks):
    # Drop rows with missing visitorid or itemid
    chunk = chunk.dropna(subset=['visitorid', 'itemid'])

    # Map event type to interaction strength
    chunk['interaction'] = chunk['event'].map(event_strength).fillna(0).astype(int)

    # Filter out interactions with zero strength (if any)
    chunk = chunk[chunk['interaction'] > 0]

    # Keep only relevant columns
    chunk = chunk[['visitorid', 'itemid', 'interaction']]

    interaction_chunks.append(chunk)

    if i == 4:  # Use only first 5 chunks to limit memory usage
        break

# Concatenate chunks into a single DataFrame
interactions_df = pd.concat(interaction_chunks)

# Build user-item interaction matrix (pivot table)
user_item_matrix = interactions_df.pivot_table(
    index='visitorid',
    columns='itemid',
    values='interaction',
    aggfunc='sum',
    fill_value=0
)

print(user_item_matrix.shape)
print(user_item_matrix.head())


(28860, 24716)
itemid     6       15      55      66      92      147     168     190     \
visitorid                                                                   
17              0       0       0       0       0       0       0       0   
52              0       0       0       0       0       0       0       0   
74              0       0       0       0       0       0       0       0   
109             0       0       0       0       0       0       0       0   
122             0       0       0       0       0       0       0       0   

itemid     195     217     ...  466760  466772  466785  466789  466795  \
visitorid                  ...                                           
17              0       0  ...       0       0       0       0       0   
52              0       0  ...       0       0       0       0       0   
74              0       0  ...       0       0       0       0       0   
109             0       0  ...       0       0       0       0       0   
1

#### Milestone 2

In [2]:
# Remove very inactive users
user_item_matrix = user_item_matrix[user_item_matrix.sum(axis=1) >= 2]

# Remove very unpopular items
user_item_matrix = user_item_matrix.loc[:, user_item_matrix.sum(axis=0) >= 2]

print("Filtered matrix shape:", user_item_matrix.shape)

Filtered matrix shape: (7297, 6036)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Compute item-item similarity
item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

print("Item similarity matrix shape:", item_similarity_df.shape)


Item similarity matrix shape: (6036, 6036)


In [4]:
def recommend_items(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found"

    user_vector = user_item_matrix.loc[user_id]

    # Score items based on similarity
    scores = item_similarity_df.dot(user_vector)

    # Remove items already interacted with
    scores[user_vector > 0] = 0

    # Sort and return top-N
    return scores.sort_values(ascending=False).head(top_n)


In [5]:
sample_user = user_item_matrix.index[0]
recommend_items(sample_user, top_n=5)


,0
itemid,
140461,0.230089
310764,0.000000
310754,0.000000
310725,0.000000
310720,0.000000


##### Tuning

In [6]:
# Remove users with zero total interaction
user_item_matrix = user_item_matrix[user_item_matrix.sum(axis=1) > 0]

# Remove items with zero total interaction
user_item_matrix = user_item_matrix.loc[:, user_item_matrix.sum(axis=0) > 0]

print("After zero-removal shape:", user_item_matrix.shape)


After zero-removal shape: (6383, 6036)


In [7]:
# -------- TUNING: NORMALIZATION --------

# Compute row sums
row_sums = user_item_matrix.sum(axis=1)

# Keep only users with non-zero interactions
user_item_matrix_nonzero = user_item_matrix[row_sums > 0]

# Normalize safely
user_item_matrix_norm = user_item_matrix_nonzero.div(
    user_item_matrix_nonzero.sum(axis=1), axis=0
)

print("NaN check:", user_item_matrix_norm.isna().sum().sum())
print("Normalized matrix shape:", user_item_matrix_norm.shape)


NaN check: 0
Normalized matrix shape: (6383, 6036)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

item_similarity_norm = cosine_similarity(user_item_matrix_norm.T)

item_similarity_norm_df = pd.DataFrame(
    item_similarity_norm,
    index=user_item_matrix_norm.columns,
    columns=user_item_matrix_norm.columns
)

print("Tuned (normalized) model trained successfully.")


Tuned (normalized) model trained successfully.


In [9]:
def recommend_items_norm(user_id, top_n=5):
    if user_id not in user_item_matrix_norm.index:
        return "User not found"

    user_vector = user_item_matrix_norm.loc[user_id]
    scores = item_similarity_norm_df.dot(user_vector)

    scores[user_vector > 0] = 0
    return scores.sort_values(ascending=False).head(top_n)


In [10]:
print("Baseline recommendations:")
print(recommend_items(sample_user))

print("\nTuned recommendations:")
print(recommend_items_norm(sample_user))


Baseline recommendations:
itemid
140461    0.230089
310764    0.000000
310754    0.000000
310725    0.000000
310720    0.000000
dtype: float64

Tuned recommendations:
itemid
140461    0.447128
310764    0.000000
310754    0.000000
310725    0.000000
310720    0.000000
dtype: float64
